# SerDes Symbol Broadening Demonstration

本Notebook演示：

1. NRZ码元生成
2. 理想信道
3. 有限带宽信道
4. 码元展宽(Symbol Broadening)
5. 码间干扰(ISI)
6. Eye Diagram


## 1 导入库

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal

plt.rcParams['figure.figsize'] = (10,4)

## 2 生成随机比特

In [ ]:
np.random.seed(0)

num_bits = 20
bits = np.random.randint(0,2,num_bits)

print(bits)

## 3 NRZ信号生成

In [ ]:
samples_per_bit = 40

tx_signal = np.repeat(bits, samples_per_bit)

plt.plot(tx_signal)
plt.title("TX NRZ Signal")
plt.xlabel("Sample")
plt.ylabel("Amplitude")
plt.grid()
plt.show()

## 4 理想信道

理想信道冲激响应：

h(t) = δ(t)

输出 = 输入

In [ ]:
ideal_channel = np.zeros(40)
ideal_channel[0] = 1

rx_ideal = np.convolve(tx_signal, ideal_channel, mode='same')

plt.plot(rx_ideal)
plt.title("RX Signal (Ideal Channel)")
plt.grid()
plt.show()

## 5 有限带宽信道

真实SerDes信道具有：

- 高频衰减
- 有限带宽

这里用一个低通滤波器模拟。

In [ ]:
b, a = signal.butter(4, 0.05)

rx_channel = signal.lfilter(b, a, tx_signal)

plt.plot(rx_channel)
plt.title("RX Signal After Channel (Symbol Broadening)")
plt.grid()
plt.show()

## 6 冲激响应

码元展宽的本质：

y(t) = x(t) * h(t)

如果h(t)有拖尾，就会导致ISI。

In [ ]:
impulse = np.zeros(200)
impulse[0] = 1

h = signal.lfilter(b, a, impulse)

plt.plot(h)
plt.title("Channel Impulse Response")
plt.grid()
plt.show()

## 7 对比 TX 与 RX

In [ ]:
plt.figure(figsize=(12,6))

plt.subplot(2,1,1)
plt.plot(tx_signal)
plt.title("TX Signal")
plt.grid()

plt.subplot(2,1,2)
plt.plot(rx_channel)
plt.title("RX Signal (ISI Appears)")
plt.grid()

plt.show()

## 8 Eye Diagram

In [ ]:
def eye_diagram(signal, sps):
    span = 2 * sps
    for i in range(0, len(signal)-span, sps):
        plt.plot(signal[i:i+span], color='blue', alpha=0.25)

plt.figure(figsize=(6,4))
eye_diagram(rx_channel, samples_per_bit)
plt.title("Eye Diagram with ISI")
plt.grid()
plt.show()

## 9 结论

码元展宽的根本原因：

信道冲激响应 h(t) 不是 δ(t)。

而是具有拖尾。

结果：

前一个码元影响后一个码元 → ISI

SerDes系统通常使用：

- TX Pre-emphasis
- RX CTLE
- DFE

来补偿ISI。